# simws 0.9a: the cryogenic core

What changes below room temperature, and how each piece is checked:

* **freeze-out** (Altermatt 2006): donors and acceptors are only partly ionized; the ionization energy falls and the
  bound fraction vanishes towards the Mott transition, so heavily doped regions stay ionized;
* **Fermi-Dirac statistics** through the degeneracy factor gamma = n / (Nc exp(eta)) (Nilsson's inverse): enters the
  space charge, the Scharfetter-Gummel currents (generalized Einstein relation), the contacts and recombination;
* **band-gap narrowing**: Slotboom's narrowing is *apparent* (fitted with Boltzmann statistics); with Fermi-Dirac the
  physical narrowing is larger by -kT ln gamma(300 K) of the majority carriers, otherwise degeneracy is counted twice;
* **Klaassen mobility**: lattice, ionized-impurity, carrier-carrier scattering and clustering; it depends on the
  carriers, so it is iterated to self-consistency after each bias point (a two-way coupling inside the adapter);
* **extended precision** below 200 K: minority densities fall below 1e-300 cm^-3, where double-precision Newton
  updates stall.

Run in the `tcad` environment from `~/sim_workspace`.

In [ ]:
import numpy as np
from simws.materials import load
from simws.materials.semiconductor import intrinsic_density
from simws.tcad import cryo as C
si = load("silicon")
for T in (300.0, 150.0, 77.0):
    ni = intrinsic_density(si, T); Nc, Nv, Eg = C.band_params(si, T, ni)
    for N in (1e16, 1e17, 1e18, 1e19):
        n, *_ = C.device_equilibrium(si, np.array([N]), np.array([0.0]), np.array([ni]), T, Nc, Nv)
        print(f"T={T:5.0f} K  ND={N:.0e}  ionized {100 * n[0] / N:5.1f} %")

## Device-level checks (closed forms, detailed balance, reciprocity)

In [ ]:
from simws.validation import cryo_checks as K
print(K.resistor_equilibrium(Ts=(77.0,), dopings=(1e16, 1e17))["worst_rel_diff"])   # freeze-out vs neutrality (gate 1e-6)
print(K.einstein_graded(Ts=(77.0,))["worst_rel"])                                    # zero-bias current / drift scale
print(K.statistics_error()["max_density_error"])                                     # Nilsson inverse vs exact F_1/2

## The cold HBT through the hub

Below 250 K results carry `extrapolated` (manifest validity box) and `cryo_unvalidated` (no measured-device validation yet).

In [ ]:
from simws.hub.jobs import Hub
hub = Hub()
for ge in (0.0, 0.15):
    r = hub.submit("devsim.hbt_gummel", params={"case": {"ge_peak": ge}, "VBE": (np.array([1.0, 1.05]), "V"),
                                                "T": (77.0, "K"), "cryo": True})
    print(ge, r.status, r.flags[:3])
    if r.status == "ok":
        print("  IC [A/m]", r.q("hbt.IC").array(), " beta", r.q("hbt.beta").array())

## Paper fidelity: mesh error bars

`fidelity="paper"` runs three meshes (spacings x1, 1/sqrt2, 1/2) and attaches the GCI of the finest mesh to every quantity as its numerical uncertainty.

In [ ]:
r = hub.submit("devsim.hbt_gummel", params={"case": {"ge_peak": 0.15}, "VBE": (np.array([0.6, 0.7]), "V"),
                                            "T": (300.0, "K")}, fidelity="paper")
q = r.q("hbt.IC"); print(q.array(), q.uncertainty.sigma)
print(r.verification["mesh"]["quantities"]["hbt.IC"])